# 02 – Zero-shot baseline ("before") and LoRA fine-tuning ("after")
Model: `Qwen/Qwen2.5-1.5B-Instruct`, fp16, greedy decoding. **Only train and val are used here. Never evaluate on test in this notebook.**
Checkpoints go to Google Drive so a disconnect does not lose them.

In [ ]:
# ==== EDIT THESE TWO LINES ====
GITHUB_USER = "FatimaRana50"
GIT_NAME    = "Fatima Rana"
# ==============================
REPO = "trialmatch"

!pip -q install peft datasets
!pip -q uninstall -y torchao       # Colab ships torchao 0.10; peft refuses to load with it (we do not use it)
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
token = userdata.get('GH_TOKEN')          # Colab sidebar -> key icon -> add secret GH_TOKEN
if not os.path.exists(f'/content/{REPO}'):
    !git clone -q https://{token}@github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git pull -q
!git config user.name "{GIT_NAME}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
import sys; sys.path.insert(0, '/content/trialmatch/src')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
CKPT_DIR = "/content/drive/MyDrive/trialmatch/lora_r16"   # change the name for each new run/config
os.makedirs(CKPT_DIR, exist_ok=True)

## 1. Baseline: base model, zero-shot, validation split (~3–5 min)

In [ ]:
!python src/evaluate.py --split val --tag base

In [ ]:
import json
recs = [json.loads(l) for l in open("results/base_val_preds.jsonl")]
for r in recs[:5]:
    print(r["criterion_type"], "| gold:", r["gold_label"], r["gold_evidence"], "| pred:", r["pred_label"], r["pred_evidence"])
    print("   raw:", r["raw_output"][:200].replace("\n", " "))

In [ ]:
!git add results/base_val_* && git commit -m "Zero-shot baseline on val" && git push

## 2. LoRA fine-tuning (~20–30 min on a T4)
Defaults: r=16, alpha=32, dropout=0.05, all attention + MLP projections, lr 2e-4 cosine, 3 epochs, effective batch 16, fp16, gradient checkpointing.
Loss is computed only on the assistant JSON. The best checkpoint is chosen by **val loss**.
If the loss shows `nan`, rerun with `--fp32` added.

In [ ]:
!python src/train.py --out {CKPT_DIR}

In [ ]:
from IPython.display import Image, display
display(Image("results/training_curves.png"))
print(open("logs/train_config.json").read())

## 3. Fine-tuned model on validation

In [ ]:
!python src/evaluate.py --split val --tag lora --adapter {CKPT_DIR}/final_adapter

## 4. Before vs after (validation)

In [ ]:
import pandas as pd
rows = []
for tag in ["base", "lora"]:
    m = json.load(open(f"results/{tag}_val_metrics.json"))
    rows.append({"model": tag,
                 "acc": m["overall"]["classification"]["accuracy"],
                 "macroF1": m["overall"]["classification"]["macro_f1"],
                 "incl_acc": m["inclusion"]["classification"]["accuracy"],
                 "excl_acc": m["exclusion"]["classification"]["accuracy"],
                 "evid_micro_F1": m["overall"]["evidence"]["micro_f1"],
                 "evid_macro_F1": m["overall"]["evidence"]["macro_f1"],
                 "valid_json": m["provenance"]["valid_json_rate"],
                 "invalid": m["overall"]["classification"]["invalid_outputs"]})
cmp = pd.DataFrame(rows).round(3); print(cmp.to_string(index=False))
cmp.to_csv("results/before_after_val.csv", index=False)

In [ ]:
!pip freeze > logs/colab_env_02.txt
!git add results/ logs/ && git commit -m "LoRA fine-tune + val comparison" && git push